<a href="https://colab.research.google.com/github/MohdFuzailHaider/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
from huggingface_hub import hf_hub_download

fact_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

print(fact_path)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [13]:
dim_content_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=HF_TOKEN,
)

print(dim_content_path)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet


In [14]:
import pandas as pd

fact_df = pd.read_parquet(fact_path)
dim_content_df = pd.read_parquet(dim_content_path)

In [15]:
import duckdb

con = duckdb.connect()

con.register("fact", fact_df)
con.register("dim_content", dim_content_df)

In [16]:
print(fact_df.shape)
print(dim_content_df.shape)
print(fact_df.columns.tolist())
print(dim_content_df.columns.tolist())

(9841378, 30)
(519606, 26)
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_pu

In [17]:
fact_df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* One row represents the daily performance of one content page (`content_hash_id`).
* For one client (`client_hash_id`) on a specific reporting date (`report_date`).
* This notebook uses data from **March 2026** (`month=2026-03`) as a min-panel month for feature engineering and verification.
* Following the assignment guidance, the final month (June 2026) is excluded from development to avoid using future outcome information.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature**
* gsc_impressions                  
* gsc_clicks
* gsc_avg_position
* ga4_pageviews
* ga4_engaged_sessions\

**Label**
* No label exists in this warehouse table.
* A proxy label will be created later during feature engineering for ranking content pages.\

**Content**
* report_date
* client_hash_id
* content_hash_id
* client_has_gsc
* client_has_ga4
* gsc_data_available
* ga4_data_available\

**Excluded**
* Client_hash_id and content_hash_id (identifiers, not predictive features).
* Any future outcome variables (to prevent data leakage).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM fact
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [24]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM fact;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [25]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS gsc_available_rows,
    SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS ga4_available_rows
FROM fact;
""").df()

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061.0,413966.0


* Search Console data is available for 3,611,061 records, while GA4 data is available for 413,966 records.
* This indicates that not all observations include data from both sources, which should be considered during feature engineering and missing-value handling.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* This dataset contains daily Search Console and GA4 performance metrics but does not directly capture external factors such as Google algorithm updates, competitor activity, or seasonality that may influence content performance.
* In addition, not all records have Search Console or GA4 data available, resulting in missing values for some engagement metrics.
* Finally, this notebook uses only the March 2026 partition, so it does not capture longer-term historical trends.

## Self-check

Before you submit, confirm each line honestly:

- [Yes] Every section above is filled — markdown thinking AND the code that backs it
- [Yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Yes] No client names, URLs, or private queries anywhere
- [Yes] My claims use careful words: observed, measured, directional, decision-support
- [Yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.